In [1]:
import os
from pathlib import Path

# Asegurar que el kernel corre desde notebooks/ para que ../data y ../outputs funcionen
_cwd = Path.cwd()
if _cwd.name != "notebooks":
    _nb_dir = _cwd / "notebooks"
    if _nb_dir.is_dir():
        os.chdir(_nb_dir)

# Crear directorios si no existen
for _d in ["../outputs", "../data", "../models"]:
    Path(_d).mkdir(exist_ok=True)

print(f"cwd: {Path.cwd()}")

cwd: c:\Users\adria\OneDrive\Desktop\Proyecto de machine learning\notebooks


# Fase 2 — Transcripciones de Earnings Calls

En esta fase cargamos el texto real de las earnings calls desde Hugging Face.

**Objetivo:** Obtener un DataFrame donde cada fila sea un fragmento de transcripción con su empresa y fecha, listo para ser analizado con NLP en la Fase siguiente.

## 1. Importar librerías

In [2]:
from datasets import load_dataset
import pandas as pd

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


## 2. Descargar el dataset desde Hugging Face

Hugging Face es como el "GitHub de los modelos e datasets de ML". Tiene miles de datasets públicos y gratuitos.

Vamos a usar `lamini/earnings-calls-qa`, un dataset con preguntas y respuestas extraídas de earnings calls reales de empresas del S&P500.

La función `load_dataset` descarga todo automáticamente, igual que `pip install` pero para datos.

In [3]:
import os
os.environ['HF_DATASETS_OFFLINE'] = '1'  # carga desde caché local, sin conectarse a internet

dataset = load_dataset('lamini/earnings-calls-qa', split='train')

print(f'Tipo de objeto: {type(dataset)}')
print(f'Cantidad de registros: {len(dataset)}')
print(f'Columnas disponibles: {dataset.column_names}')

Tipo de objeto: <class 'datasets.arrow_dataset.Dataset'>
Cantidad de registros: 860164
Columnas disponibles: ['question', 'answer', 'date', 'transcript', 'q', 'ticker', 'predictions']


## 3. Explorar la estructura del dataset

Antes de usar cualquier dataset hay que entender qué contiene. Miramos las primeras filas para ver el formato del texto.

In [4]:
# Convertimos a DataFrame de pandas para trabajar más cómodo
df_raw = dataset.to_pandas()
print(f'Shape: {df_raw.shape}')
print(f'\nTipos de columnas:')
print(df_raw.dtypes)
df_raw.head(3)

Shape: (860164, 7)

Tipos de columnas:
question          str
answer            str
date              str
transcript        str
q                 str
ticker            str
predictions    object
dtype: object


,question,answer,date,transcript,q,ticker,predictions
0,What was TSMC's revenue in US dollar terms in ...,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
1,What was TSMC's EPS in 2019,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."
2,What was TSMC's capex spending in 2019,I do not know. The transcript does not provid...,"Jan 13, 2022, 1:00 a.m. ET",and for our industry-leading advanced and spec...,2021-Q4,TSM,"[{'class_id': 0, 'class_name': 'correct', 'pro..."


In [5]:
# Miramos un ejemplo completo para entender el texto real
print('=== EJEMPLO DE REGISTRO ===\n')
ejemplo = df_raw.iloc[0]
for col in df_raw.columns:
    valor = str(ejemplo[col])
    print(f'[{col}]')
    print(valor[:500])
    print()

=== EJEMPLO DE REGISTRO ===

[question]
What was TSMC's revenue in US dollar terms in 2019 

[answer]
 I do not know. The transcript does not provide the revenue for TSMC in US dollar terms in 2019.

[date]
Jan 13, 2022, 1:00 a.m. ET

[transcript]
and for our industry-leading advanced and specialty technologies, where we see strong interest from all four growth platforms, which are smartphone, HPC, IoT, and automotive. Entering 2022, we expect the supply chain to maintain a higher level of inventory as compared to the historical seasonal level given the industry's continued need to ensure supply security.
While the short-term imbalance may or may not persist, we continue to observe the structural increase in long-term semiconductor demand

[q]
2021-Q4

[ticker]
TSM

[predictions]
[{'class_id': 0, 'class_name': 'correct', 'prob': 0.7940037696689208}
 {'class_id': 1, 'class_name': 'incorrect', 'prob': 0.2059962303310792}]



## 4. Limpiar y preparar el texto

El texto crudo de las transcripciones tiene ruido: espacios extra, saltos de línea, caracteres especiales. Lo limpiamos antes de pasarlo al modelo NLP.

También vamos a quedarnos con la columna de texto principal — la que contiene el fragmento de la earnings call.

In [6]:
import re

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ''
    texto = re.sub(r'\s+', ' ', texto)
    texto = re.sub(r'\n+', ' ', texto)
    texto = texto.strip()
    return texto

# 12 empresas: 5 originales + 7 nuevas (TSLA, AMZN, NVDA, AMD, JPM, V, MA)
tickers_objetivo = ['AAPL', 'MSFT', 'GOOGL', 'META', 'NFLX', 'TSLA', 'AMZN', 'NVDA', 'AMD', 'JPM', 'V', 'MA']

# Filtramos solo las empresas que nos interesan
df_filtrado = df_raw[df_raw['ticker'].isin(tickers_objetivo)].copy()
print(f'Registros de nuestras empresas: {len(df_filtrado)}')
print(f'Distribución por ticker:\n{df_filtrado["ticker"].value_counts()}\n')

# El dataset repite la transcripción una vez por cada pregunta
# Nos quedamos con una sola fila por transcripción única (ticker + trimestre)
df_unique = df_filtrado.drop_duplicates(subset=['ticker', 'q'])[['ticker', 'date', 'q', 'transcript']].copy()
print(f'Transcripciones únicas (una por trimestre): {len(df_unique)}')

# Limpiamos el texto
df_unique['texto'] = df_unique['transcript'].apply(limpiar_texto)
df_unique = df_unique.drop(columns=['transcript'])
df_unique = df_unique[df_unique['texto'].str.len() > 100].reset_index(drop=True)

print(f'\nLongitud promedio del texto: {df_unique["texto"].str.len().mean():.0f} caracteres')
print(f'\nEjemplo:')
print(f'Ticker: {df_unique.iloc[0]["ticker"]} | Trimestre: {df_unique.iloc[0]["q"]}')
print(df_unique.iloc[0]['texto'][:400])

Registros de nuestras empresas: 18018
Distribución por ticker:
ticker
AAPL     4138
GOOGL    3880
TSLA     3553
AMZN     1893
META      870
JPM       662
AMD       631
MA        602
V         510
NFLX      452
NVDA      446
MSFT      381
Name: count, dtype: int64

Transcripciones únicas (una por trimestre): 92

Longitud promedio del texto: 4030 caracteres

Ejemplo:
Ticker: AAPL | Trimestre: 2020-Q4
the reception that we've gotten so far, we're very confident there. Shannon Cross -- Cross Research -- Analyst OK, great. And then, can you talk a bit about just overall in the world -- the cadence that you see sort of for the 5G adoption launch? What you see will be sort of the key drivers? Obviously, there's a fair amount of subsidies going on in the U.S. at this point. Thank you. Tim Cook -- Ch


## 5. Guardar las transcripciones limpias

Guardamos el resultado para usarlo en la Fase 3 (análisis con FinBERT).

In [7]:
df_unique.to_csv('../data/transcripciones_clean.csv', index=False)
print(f'Guardado en data/transcripciones_clean.csv')
print(f'Shape final: {df_unique.shape}')
print(f'\nVista previa:')
df_unique[['ticker', 'date', 'q']].head(10)

Guardado en data/transcripciones_clean.csv
Shape final: (92, 4)

Vista previa:


,ticker,date,q
0,AAPL,"Oct 29, 2020, 5:00 p.m. ET",2020-Q4
1,NVDA,"Nov 14, 2019, 5:30 p.m. ET",2019-Q3
2,GOOGL,"Oct 26, 2021, 4:30 p.m. ET",2021-Q3
3,GOOGL,"Apr 27, 2021, 5:00 p.m. ET",2021-Q1
4,AMZN,"Feb 02, 2021, 5:30 p.m. ET",2020-Q4
5,META,"Apr 27, 2022, 5:00 p.m. ET",2022-Q1
6,NVDA,"Feb 24, 2021, 5:00 p.m. ET",2021-Q4
7,AAPL,"Jan 28, 2020, 5:00 p.m. ET",2020-Q1
8,NVDA,"Aug 15, 2019, 5:30 p.m. ET",2020-Q2
9,TSLA,"Apr 29, 2020, 6:30 p.m. ET",2020-Q1
